# User Churn Prediction and Behavioral Analysis

## Project Objective
Analyze user behavior in the Sparkify event dataset, identify behavioral patterns associated with churn, and build an interpretable model to estimate future churn risk.

### Key principle
The model must predict **future churn** using only behavior observed **before the prediction cutoff**. This prevents data leakage.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)

# Place the dataset in the data/ folder (not included in the repo).
DATA_PATH = "data/mini_sparkify_event_data.json"

df = pd.read_json(DATA_PATH, lines=True)
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

## 2. Data Understanding

Each row represents a user event or interaction. The `page` column identifies the type of event.

For this project, **`Cancellation Confirmation` is the churn event**.  
`Downgrade` is treated as a subscription change, not as churn.

In [ ]:
print("Event types:")
print(df["page"].value_counts())

print("\nAuthentication status:")
print(df["auth"].value_counts())

In [ ]:
# Convert Unix timestamps from milliseconds to datetime.
df["event_time"] = pd.to_datetime(df["ts"], unit="ms")

churn_events = df[df["page"] == "Cancellation Confirmation"].copy()

print("Dataset period:", df["event_time"].min(), "to", df["event_time"].max())
print("Users with non-empty IDs:", df.loc[df["userId"] != "", "userId"].nunique())
print("Observed churners:", churn_events["userId"].nunique())

## 3. Churn Definition and Prediction Cutoff

In [ ]:
# Remove events that do not belong to an identifiable user.
model_data = df[df["userId"] != ""].copy()

# Prediction cutoff: only behavior before this date is used as model input.
cutoff = pd.Timestamp("2018-11-01")

# Users who had already churned before the cutoff are excluded.
churned_before_cutoff = set(
    churn_events.loc[churn_events["event_time"] < cutoff, "userId"]
)

# Users who had activity before the cutoff are eligible for prediction.
active_before_cutoff = set(
    model_data.loc[model_data["event_time"] < cutoff, "userId"]
)

eligible_users = active_before_cutoff - churned_before_cutoff

# Future churn: cancellation on or after the cutoff.
future_churn = (
    churn_events.loc[
        churn_events["event_time"] >= cutoff, ["userId"]
    ]
    .drop_duplicates()
)

future_churn["churn"] = 1

target = (
    pd.DataFrame(index=sorted(eligible_users))
    .rename_axis("userId")
    .join(future_churn.set_index("userId"), how="left")
    .fillna({"churn": 0})
    .astype({"churn": int})
)

print("Eligible users:", len(eligible_users))
print("Future churners:", target["churn"].sum())
print("Future non-churners:", (target["churn"] == 0).sum())
print("Churn rate among eligible users:", round(target["churn"].mean() * 100, 2), "%")

### Why the cutoff matters

A user's behavior after the prediction date must not be used to predict whether that user will churn. Otherwise, the model can indirectly see the future outcome and produce misleadingly strong results.

## 4. Exploratory Data Analysis

In [ ]:
# Class distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=target.reset_index(), x="churn")
plt.title("Future Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Users")
plt.show()

In [ ]:
# Build a reusable pre-cutoff dataset for EDA and modeling.
pre_cutoff_data = model_data[
    (model_data["event_time"] < cutoff) &
    (model_data["userId"].isin(eligible_users))
].copy()

print("Pre-cutoff events:", len(pre_cutoff_data))
print("Pre-cutoff users:", pre_cutoff_data["userId"].nunique())

In [ ]:
# Session-level statistics
session_stats = (
    pre_cutoff_data
    .groupby(["userId", "sessionId"])["event_time"]
    .agg(["min", "max"])
)

session_stats["duration_min"] = (
    session_stats["max"] - session_stats["min"]
).dt.total_seconds() / 60

session_features = (
    session_stats
    .groupby("userId")["duration_min"]
    .agg(
        avg_session_duration="mean",
        median_session_duration="median",
        num_sessions="count"
    )
)

song_features = (
    pre_cutoff_data.groupby("userId")["page"]
    .agg(songs_played=lambda x: (x == "NextSong").sum())
)

eda_engagement = song_features.join(session_features).join(target)

eda_engagement.groupby("churn")[
    ["songs_played", "num_sessions",
     "avg_session_duration", "median_session_duration"]
].median()

In [ ]:
# Median session gap is used for EDA only; it is not included in the final model.
session_gap = session_stats.copy()
session_gap["days_since_previous_session"] = (
    session_gap.groupby(level=0)["min"]
    .diff()
    .dt.total_seconds() / (60 * 60 * 24)
)

median_session_gap = (
    session_gap.groupby(level=0)["days_since_previous_session"]
    .median()
    .rename("median_session_gap")
)

median_session_gap.to_frame().join(target).groupby("churn")[
    "median_session_gap"
].median()

In [ ]:
# Feedback behavior
feedback_eda = (
    pre_cutoff_data.groupby("userId")["page"]
    .agg(
        thumbs_up=lambda x: (x == "Thumbs Up").sum(),
        thumbs_down=lambda x: (x == "Thumbs Down").sum()
    )
)

feedback_eda["thumbs_down_rate"] = (
    feedback_eda["thumbs_down"] /
    (feedback_eda["thumbs_up"] + feedback_eda["thumbs_down"])
).fillna(0)

feedback_eda.join(target).groupby("churn")[
    ["thumbs_up", "thumbs_down", "thumbs_down_rate"]
].median()

In [ ]:
# Social and subscription behavior
social_eda = (
    pre_cutoff_data.groupby("userId")["page"]
    .agg(
        friend_adds=lambda x: (x == "Add Friend").sum(),
        playlist_adds=lambda x: (x == "Add to Playlist").sum()
    )
)

subscription_eda = (
    pre_cutoff_data.groupby("userId")["page"]
    .agg(
        upgrades=lambda x: (x == "Upgrade").sum(),
        downgrades=lambda x: (x == "Downgrade").sum()
    )
)

print("Social behavior:")
print(social_eda.join(target).groupby("churn").median())

print("\nSubscription behavior:")
print(subscription_eda.join(target).groupby("churn").median())

In [ ]:
# Content diversity
music_eda = (
    pre_cutoff_data.groupby("userId")
    .agg(
        unique_artists=("artist", "nunique"),
        unique_songs=("song", "nunique")
    )
)

music_eda.join(target).groupby("churn")[
    ["unique_artists", "unique_songs"]
].median()

In [ ]:
# Four main EDA comparisons
plot_data = eda_engagement.join(feedback_eda).join(social_eda).join(
    subscription_eda
).join(music_eda)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.boxplot(data=plot_data, x="churn", y="songs_played", ax=axes[0, 0])
axes[0, 0].set_title("Songs Played vs Churn")

sns.boxplot(data=plot_data, x="churn", y="num_sessions", ax=axes[0, 1])
axes[0, 1].set_title("Number of Sessions vs Churn")

sns.boxplot(data=plot_data, x="churn", y="thumbs_down_rate", ax=axes[1, 0])
axes[1, 0].set_title("Thumbs-Down Rate vs Churn")

sns.boxplot(data=plot_data, x="churn", y="downgrades", ax=axes[1, 1])
axes[1, 1].set_title("Downgrades vs Churn")

plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Core engagement features
pre_features = (
    song_features
    .join(session_features.drop(columns="num_sessions"))
    .join(feedback_eda)
    .join(social_eda)
    .join(subscription_eda)
    .join(music_eda)
    .join(target)
)

# Current subscription level = the user's most recent level before the cutoff.
level_data = pre_cutoff_data.sort_values(["userId", "event_time"])

current_level = (
    level_data.groupby("userId")["level"]
    .last()
    .rename("current_level")
)

pre_features = pre_features.join(current_level)

# Derived feature: total subscription changes.
pre_features["subscription_changes"] = (
    pre_features["upgrades"] + pre_features["downgrades"]
)

pre_features.head()

In [ ]:
print("Feature shape:", pre_features.shape)
print("Missing values:")
print(pre_features.isna().sum())

print("\nFeatures:")
print(pre_features.columns.tolist())

## 6. Feature Correlation and Redundancy

In [ ]:
numeric_features_eda = pre_features.select_dtypes(include="number")
corr = numeric_features_eda.corr()

print("Correlation with churn:")
print(corr["churn"].sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

`subscription_changes` is removed because it is directly calculated as `upgrades + downgrades`, making it largely redundant with those two features.

`unique_songs` is removed later after comparing the reduced model with the original model; removing it does not change the cross-validation results.

In [ ]:
pre_features = pre_features.drop(columns=["subscription_changes"])

## 7. Train/Test Split and Preprocessing

In [ ]:
X = pre_features.drop(columns=["churn"])
y = pre_features["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

categorical_features = ["current_level"]
numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Training users:", len(X_train))
print("Test users:", len(X_test))

## 8. Logistic Regression Baseline

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            random_state=42,
            max_iter=1000
        ))
    ]
)

logistic_model.fit(X_train, y_train)
y_pred = logistic_model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## 9. Random Forest Comparison

In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            max_depth=5
        ))
    ]
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, rf_pred))

### Why use cross-validation?

There are relatively few churners, so a single train/test split can give unstable results. Stratified 5-fold cross-validation keeps the churn/non-churn proportion approximately consistent across folds and gives a more informative comparison.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

logistic_cv = cross_validate(
    logistic_model, X, y, cv=cv, scoring=scoring
)

rf_cv = cross_validate(
    rf_model, X, y, cv=cv, scoring=scoring
)

comparison = pd.DataFrame({
    "Logistic Regression": [
        logistic_cv["test_accuracy"].mean(),
        logistic_cv["test_precision"].mean(),
        logistic_cv["test_recall"].mean(),
        logistic_cv["test_f1"].mean(),
        logistic_cv["test_roc_auc"].mean()
    ],
    "Random Forest": [
        rf_cv["test_accuracy"].mean(),
        rf_cv["test_precision"].mean(),
        rf_cv["test_recall"].mean(),
        rf_cv["test_f1"].mean(),
        rf_cv["test_roc_auc"].mean()
    ]
}, index=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])

comparison.round(3)

### Model comparison

Accuracy alone is not sufficient because churn is the minority class. Recall, F1, ROC-AUC, and later PR-AUC are more informative for evaluating churn detection.

The cross-validation results show that Logistic Regression identifies a larger share of churners and has stronger F1 and ROC-AUC performance than the Random Forest configuration used here.

## 10. Reduced Logistic Regression Model

In [ ]:
# Remove a highly redundant feature and compare performance.
X_reduced = pre_features.drop(
    columns=["churn", "unique_songs"]
)
y = pre_features["churn"]

categorical_features_reduced = ["current_level"]
numeric_features_reduced = [
    col for col in X_reduced.columns
    if col not in categorical_features_reduced
]

preprocessor_reduced = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features_reduced),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_reduced)
    ]
)

logistic_reduced = Pipeline(
    steps=[
        ("preprocessor", preprocessor_reduced),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [ ]:
reduced_cv = cross_validate(
    logistic_reduced,
    X_reduced,
    y,
    cv=cv,
    scoring=scoring
)

reduced_results = pd.Series({
    "Accuracy": reduced_cv["test_accuracy"].mean(),
    "Precision": reduced_cv["test_precision"].mean(),
    "Recall": reduced_cv["test_recall"].mean(),
    "F1": reduced_cv["test_f1"].mean(),
    "ROC-AUC": reduced_cv["test_roc_auc"].mean()
})

reduced_results.round(3)

The reduced Logistic Regression model gives essentially the same cross-validation performance while using fewer predictors. This keeps the final model simpler and easier to explain.

## 11. Cross-Validated Probabilities and Threshold Analysis

In [ ]:
# Generate out-of-fold probabilities from the final reduced model.
y_prob = cross_val_predict(
    logistic_reduced,
    X_reduced,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for threshold in thresholds:
    y_threshold = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y, y_threshold, zero_division=0),
        "recall": recall_score(y, y_threshold, zero_division=0),
        "f1": f1_score(y, y_threshold, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df.round(3)

A lower threshold catches more potential churners but also creates more false positives. A higher threshold produces fewer positive predictions but can miss more churners.

For the final demonstration, a threshold of **0.60** is used because it was the threshold examined in the project analysis. Since the threshold was selected using the same cross-validated predictions used for evaluation, it should be treated as an analytical choice rather than an unbiased production threshold.

In [ ]:
FINAL_THRESHOLD = 0.60

y_final = (y_prob >= FINAL_THRESHOLD).astype(int)

print("Final threshold:", FINAL_THRESHOLD)
print("Precision:", round(precision_score(y, y_final), 3))
print("Recall:", round(recall_score(y, y_final), 3))
print("F1:", round(f1_score(y, y_final), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y, y_final))

## 12. Final Evaluation Plots

In [ ]:
# Confusion matrix
cm = confusion_matrix(y, y_final)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"]
)
plt.title(f"Confusion Matrix — Threshold {FINAL_THRESHOLD}")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y, y_prob)
roc_auc = roc_auc_score(y, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

print("ROC-AUC:", round(roc_auc, 3))

In [ ]:
# Precision-Recall curve
precision, recall, _ = precision_recall_curve(y, y_prob)
pr_auc = average_precision_score(y, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"Average Precision = {pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.show()

print("PR-AUC / Average Precision:", round(pr_auc, 3))

## 13. Feature Interpretation

In [ ]:
# Fit the final reduced model on all eligible users for interpretation.
logistic_reduced.fit(X_reduced, y)

feature_names = (
    logistic_reduced
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    logistic_reduced
    .named_steps["classifier"]
    .coef_[0]
)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
}).sort_values("coefficient")

coef_df

In [ ]:
plt.figure(figsize=(10, 7))
plt.barh(coef_df["feature"], coef_df["coefficient"])
plt.axvline(0, linestyle="--")
plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.title("Features Associated with Predicted Churn")
plt.tight_layout()
plt.show()

### How to read the coefficients

- A **positive** coefficient means higher values are associated with higher predicted churn probability, holding the other model variables constant.
- A **negative** coefficient means higher values are associated with lower predicted churn probability, holding the other model variables constant.
- These are **associations, not causal effects**.
- Because the dataset is small and several behavioral variables are correlated, coefficient magnitudes should be interpreted cautiously.

## 14. Business Insights

Based on the analysis in this notebook:

1. **Churn is not simply a low-activity problem.** In this dataset, churned users can show substantial engagement, including more sessions and songs played.
2. **Negative feedback is informative.** The thumbs-down rate shows a positive association with churn in the feature analysis.
3. **Subscription behavior matters.** Upgrades and downgrades provide information about user subscription activity.
4. **Content diversity differs between groups.** Unique artists and unique songs show differences between churned and non-churned users.
5. **Multiple behavioral signals should be considered together.** No single feature should be interpreted as a complete explanation of churn.

## 15. Limitations

- The dataset contains relatively few churners, so model estimates can be unstable.
- The class distribution is imbalanced.
- The data are observational; model associations do not establish causation.
- The final threshold was examined using cross-validated predictions from the same dataset, so it should not be treated as a production-validated threshold.
- A future time-period holdout would provide a stronger test of real-world generalization.
- Additional users and churn events would support more reliable model training and evaluation.

## 16. Conclusion

This project analyzes user behavior before a fixed prediction cutoff and uses those features to estimate future churn. The reduced Logistic Regression model provides an interpretable baseline and captures churn-related behavioral patterns while avoiding post-cutoff information leakage.

The main takeaway is that churn in this dataset is associated with a combination of engagement, feedback, subscription, social, and content-diversity signals rather than a single measure of inactivity.